# WellAware AI - Phase 2: Model Selection & Pretrained Pipeline
**Project:** WellAware AI: Patient Wellness Campaign Visual Generator  
**Objective:** Verify GPU runtime, load `runwayml/stable-diffusion-v1-5` via Hugging Face Diffusers, and generate a baseline patient wellness awareness visual.

### Step 1: Verify Hardware & GPU Acceleration
Stable Diffusion requires an NVIDIA GPU for fast inference. Let's confirm that Google Colab has allocated a T4 GPU.

In [ ]:
import torch

print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[SUCCESS] GPU Detected: {gpu_name}")
    print(f"[SUCCESS] Available VRAM: {vram:.2f} GB")
else:
    print("[WARNING] No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)

### Step 2: Install Required Libraries in Colab
We install `diffusers`, `transformers`, `accelerate`, and `safetensors`.

In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors
print("[SUCCESS] Dependencies installed and upgraded cleanly!")

### Step 3: Load the Pretrained Stable Diffusion Model
- **Model:** `runwayml/stable-diffusion-v1-5`
- **Precision:** `torch.float16` for ~50% VRAM reduction and 2x faster inference.
- **Scheduler:** `DPMSolverMultistepScheduler` (high quality output in 25 steps instead of 50).
- **Memory Optimization:** `pipe.enable_attention_slicing()` to ensure smooth memory management.

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch

MODEL_ID = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_ID} on device: {device}...")

# Load pipeline with half-precision (float16) on GPU
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None  # Can be enabled or kept default; using clean pipeline
)

# Use DPM-Solver++ for faster, sharper convergence (25 steps instead of 50)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

# Memory optimization for GPU
if device == "cuda":
    pipe.enable_attention_slicing()

print("[SUCCESS] Pretrained Stable Diffusion pipeline loaded successfully!")

### Step 4: Define Campaign Prompt & Generate Wellness Visual
We will generate a sample campaign visual for **Heart Health Awareness**.

In [ ]:
import os

# Health campaign prompts
prompt = (
    "A professional healthcare wellness campaign poster for heart health awareness, "
    "a vibrant healthy heart symbol surrounded by fresh organic fruits, running shoes, and a medical stethoscope, "
    "clean minimalist composition, soft natural morning sunlight, modern flat 3D illustration style, "
    "calming teal and warm coral color palette, 8k resolution, elegant, uplifting atmosphere"
)

negative_prompt = (
    "ugly, deformed, disfigured, blurry, low resolution, pixelated, bad anatomy, "
    "gory, morbid, blood, dark, terrifying, text, watermark, signature, cut off"
)

# Deterministic generator seed for reproducibility
generator = torch.Generator(device=device).manual_seed(42)

print("Generating wellness campaign visual...")

output = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=25,
    guidance_scale=7.5,
    width=512,
    height=512,
    generator=generator
)

generated_image = output.images[0]
output_filename = "heart_health_campaign.png"
generated_image.save(output_filename)
print(f"[SUCCESS] Image saved as '{output_filename}'! Size: {generated_image.size}")

### Step 5: Display the Generated Visual in Colab

In [ ]:
from IPython.display import display

# Display inside the notebook
display(generated_image)